# SAE Training on Federated Learning Checkpoints

In [ ]:
# Cell 1: Environment Bootstrap
import os
REPO_URL = "https://github.com/ha405/FedMI.git"
BRANCH = "cvpr"

if not os.path.exists("FedMI"):
    get_ipython().system(f"git clone -b {BRANCH} {REPO_URL}")
else:
    # Update repo if already exists to get latest fixes
    get_ipython().system(f"cd FedMI && git pull origin {BRANCH}")

os.chdir("FedMI")

from fedmi.env import setup, CHECKPOINT_DIR, DATA_DIR
setup()

# Install dependencies required by the original notebook
%pip install -q einops opencv-python scikit-image overcomplete

In [ ]:
import torch
import torch.nn as nn
from typing import List
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange
from torch.utils.data import DataLoader, TensorDataset
import json
import os

import overcomplete
from overcomplete.sae import TopKSAE, JumpSAE, BatchTopKSAE, train_sae
from overcomplete.visualization import show, overlay_top_heatmaps

import sys
# sys.path.append('..') # Original path, now unnecessary as we are in the repo root

from core.config import ExperimentConfig
from core.dataset import get_dataset, get_test_dataloader

class SimpleCNN(nn.Module):
    def __init__(self, conv_channels: List[int] = None, num_classes: int = 10, input_channels: int = 1):
        super(SimpleCNN, self).__init__()
        if conv_channels is None:
            conv_channels = [32, 64, 128]
        
        self.conv1 = nn.Conv2d(input_channels, conv_channels[0], 3, padding=1)
        self.conv2 = nn.Conv2d(conv_channels[0], conv_channels[1], 3, padding=1)
        self.conv3 = nn.Conv2d(conv_channels[1], conv_channels[2], 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        self.spatial_dim = 3 if input_channels == 1 else 4
        self.fc = nn.Linear(conv_channels[2] * self.spatial_dim * self.spatial_dim, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

def get_model(config) -> nn.Module:
    in_channels = 1 if config.dataset_name == "MNIST" else 3
    return SimpleCNN(
        conv_channels=config.conv_channels,
        num_classes=config.num_classes,
        input_channels=in_channels
    ).to(config.device)

def load_config_and_data(config_path):
    with open(config_path, 'r') as f:
        config_dict = json.load(f)
    # Filter out comment keys
    config_dict = {k: v for k, v in config_dict.items() if not k.startswith('_')}
    config = ExperimentConfig(**config_dict)
    # Patch paths for Cloud environment
    config.data_root = DATA_DIR
    _, testset = get_dataset(config)
    test_loader = get_test_dataloader(testset, config)
    return config, testset, test_loader

def load_model_from_checkpoint(config, path):
    model = get_model(config)
    state_dict = torch.load(path, map_location=config.device)
    model.load_state_dict(state_dict['model_state_dict'])
    model.eval()
    return model

def extract_activations(model, dataloader, config):
    model.eval()
    activations = []
    labels = []
    with torch.no_grad():
        for batch in dataloader:
            x, y = batch
            x = x.to(config.device)
            x = model.pool(torch.relu(model.conv1(x)))
            x = model.pool(torch.relu(model.conv2(x)))
            b, c, h, w = x.shape
            x_flat = rearrange(x, 'b c h w -> (b h w) c')
            activations.append(x_flat.cpu())
            y_rep = y.unsqueeze(1).repeat(1, h*w).reshape(-1)
            labels.append(y_rep.cpu())
    activations = torch.cat(activations, dim=0)
    labels = torch.cat(labels, dim=0)
    print("Activation shape:", activations.shape)
    print("Label shape:", labels.shape)
    return activations, labels

In [ ]:
def train_jump_sae(activations, scenario_name, config):
    subset_size = min(10000, len(activations))
    activations_subset = activations[:subset_size]
    dataset = TensorDataset(activations_subset.to(config.device))
    dataloader = DataLoader(dataset, batch_size=512, shuffle=True)
    
    print(f"\n--- Training JumpReLU SAE for {scenario_name} ---")
    print(f"Using {subset_size} samples of size {activations.shape[-1]} for SAE training.")

    num_features = activations.shape[-1]
    sae_jump = JumpSAE(num_features, nb_concepts=num_features * 4, bandwidth=1e-2, kernel='silverman', device=config.device)
    optimizer_jump = torch.optim.Adam(sae_jump.parameters(), lr=3e-3)
    
    desired_sparsity = 0.10
    
    def criterion_jump(x, x_hat, pre_codes, codes, dictionary):
        loss = (x - x_hat).square().mean()
        sparsity = (codes > 0).float().mean().detach()
        if sparsity > desired_sparsity:
            loss -= sae_jump.thresholds.sum()
        return loss
        
    logs_jump = train_sae(sae_jump, dataloader, criterion_jump, optimizer_jump, nb_epochs=20, device=config.device)
    
    print(f"JumpReLU SAE training completed for {scenario_name}.")
    return sae_jump

## IID Experiment

In [ ]:
config_iid, testset_iid, test_loader_iid = load_config_and_data('default_config.json')

# Load IID model - Adjusting path relative to CHECKPOINT_DIR
IID_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'iid_experiment/checkpoints/checkpoint_round_5.pt')
if not os.path.exists(IID_CHECKPOINT):
    print(f"Checking fallback path for IID...")
    IID_CHECKPOINT = 'checkpoints/iid_5class/checkpoints/checkpoint_round_5.pt'

if os.path.exists(IID_CHECKPOINT):
    model_iid = load_model_from_checkpoint(config_iid, IID_CHECKPOINT)
    # Extract activations from 2nd layer
    activations_iid, labels_iid = extract_activations(model_iid, test_loader_iid, config_iid)
    print(f"IID Activations shape: {activations_iid.shape}")
    # Train JumpReLU SAE
    sae_iid = train_jump_sae(activations_iid, "IID", config_iid)
else:
    print(f"\u26a0\ufe0f IID checkpoint not found at {IID_CHECKPOINT}. Ensure experiments have been run.")

## Non-IID Experiment

In [ ]:
config_niid, testset_niid, test_loader_niid = load_config_and_data('configs/non_iid_dirichlet.json')

# Load Non-IID model - Adjusting path relative to CHECKPOINT_DIR
NIID_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'niid_experiment/checkpoints/checkpoint_round_5.pt')
if not os.path.exists(NIID_CHECKPOINT):
    print(f"Checking fallback path for NIID...")
    NIID_CHECKPOINT = 'checkpoints/non_iid_dirichlet/checkpoints/checkpoint_round_5.pt'

if os.path.exists(NIID_CHECKPOINT):
    model_niid = load_model_from_checkpoint(config_niid, NIID_CHECKPOINT)
    # Extract activations from 2nd layer
    activations_niid, labels_niid = extract_activations(model_niid, test_loader_niid, config_niid)
    print(f"Non-IID Activations shape: {activations_niid.shape}")
    # Train JumpReLU SAE
    sae_niid = train_jump_sae(activations_niid, "Non-IID", config_niid)
else:
    print(f"\u26a0\ufe0f Non-IID checkpoint not found at {NIID_CHECKPOINT}.")

In [ ]:
print("\n==============================")
print("SAE GEOMETRY ANALYSIS START")
print("==============================")

if 'activations_iid' in dir() and 'activations_niid' in dir():
    print(f"\nIID activation shape: {activations_iid.shape}")
    print(f"Non-IID activation shape: {activations_niid.shape}")

    import torch.nn.functional as F
    from sklearn.decomposition import PCA

    # [1] Extracting dictionary feature directions
    if 'sae_iid' in dir() and 'sae_niid' in dir():
        w_iid = F.normalize(sae_iid.dictionary.weight.data.float(), dim=1)
        w_niid = F.normalize(sae_niid.dictionary.weight.data.float(), dim=1)
        print(f"\n[1] Extracting dictionary feature directions")
        print(f"IID dictionary: {w_iid.shape}")
        print(f"Non-IID dictionary: {w_niid.shape}")

        # [2] Cross-model feature similarity
        cross_sim = torch.mm(w_iid, w_niid.T)
        max_sims, _ = cross_sim.max(dim=1)
        print(f"\n[2] Cross-model feature similarity")
        print(f"Mean max similarity: {max_sims.mean().item()}")
        print(f"Median max similarity: {max_sims.median().item()}")

        # Plot cross-model similarity histogram
        plt.figure(figsize=(6, 4))
        plt.hist(max_sims.cpu().numpy(), bins=30, alpha=0.7)
        plt.title("Cross-model Best-Match Cosine Similarity")
        plt.xlabel("Cosine Similarity")
        plt.ylabel("Count")
        plt.show()

        # [3] Internal feature geometry
        print(f"\n[3] Internal feature geometry")
        def analyze_geometry(w, name):
            sim = torch.mm(w, w.T)
            mask = ~torch.eye(sim.shape[0], dtype=bool, device=sim.device)
            off_diag = sim[mask]
            print(f"{name} mean pairwise similarity: {off_diag.mean().item()}")
            return off_diag

        off_diag_iid = analyze_geometry(w_iid, "IID")
        off_diag_niid = analyze_geometry(w_niid, "Non-IID")

        # Similarity histograms side by side
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(off_diag_iid.cpu().numpy(), bins=50, alpha=0.7)
        axes[0].set_title("Pairwise Cosine Similarity - IID")
        axes[1].hist(off_diag_niid.cpu().numpy(), bins=50, alpha=0.7)
        axes[1].set_title("Pairwise Cosine Similarity - Non-IID")
        plt.tight_layout()
        plt.show()

        # [4] PCA geometry
        print(f"\n[4] PCA geometry")
        pca = PCA(n_components=2)
        coords_iid = pca.fit_transform(w_iid.cpu().numpy())
        coords_niid = pca.fit_transform(w_niid.cpu().numpy())

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].scatter(coords_iid[:, 0], coords_iid[:, 1], s=10, alpha=0.5)
        axes[0].set_title("PCA of Dictionary Directions - IID")
        axes[1].scatter(coords_niid[:, 0], coords_niid[:, 1], s=10, alpha=0.5)
        axes[1].set_title("PCA of Dictionary Directions - Non-IID")
        plt.tight_layout()
        plt.show()

        # [5] Computing SAE feature activations
        print(f"\n[5] Computing SAE feature activations")
        with torch.no_grad():
            subset_iid = activations_iid[:10000].to(w_iid.device)
            codes_iid = sae_iid.encode(subset_iid)
            active_iid = (codes_iid > 0).float().mean().item()
            print(f"IID active feature fraction: {active_iid}")

            subset_niid = activations_niid[:10000].to(w_niid.device)
            codes_niid = sae_niid.encode(subset_niid)
            active_niid = (codes_niid > 0).float().mean().item()
            print(f"Non-IID active feature fraction: {active_niid}")

        # [6] Feature entropy
        print(f"\n[6] Feature entropy")
        def feature_entropy(codes):
            """Compute normalized entropy of feature activation frequencies."""
            freq = (codes > 0).float().mean(dim=0)  # per-feature activation rate
            freq = freq + 1e-10  # avoid log(0)
            freq = freq / freq.sum()  # normalize to distribution
            entropy = -(freq * torch.log(freq)).sum()
            max_entropy = torch.log(torch.tensor(float(codes.shape[1])))
            return (entropy / max_entropy).item()

        ent_iid = feature_entropy(codes_iid)
        ent_niid = feature_entropy(codes_niid)
        print(f"IID Normalized Feature Entropy: {ent_iid:.4f}")
        print(f"Non-IID Normalized Feature Entropy: {ent_niid:.4f}")
        print("(High entropy = model uses features uniformly; Low = model relies on very few features)")

        # [7] Per-Class Projection: Non-IID activations onto IID features
        print(f"\n[7] Per-Class Projection: Non-IID activations onto IID features")
        unique_labels = labels_niid[:10000].unique()
        for lbl in sorted(unique_labels.tolist()):
            lbl_int = int(lbl)
            mask = labels_niid[:10000] == lbl
            class_acts = activations_niid[:10000][mask].to(w_iid.device)
            # Project Non-IID activations onto IID dictionary
            projections = torch.mm(class_acts, w_iid.T)
            mean_proj = projections.abs().mean().item()
            strongest_feat = projections.abs().mean(dim=0).argmax().item()
            strongest_val = projections.abs().mean(dim=0).max().item()
            print(f"Class {lbl_int} | Mean Projection: {mean_proj:.4f} | Strongest IID Feature: {strongest_feat} (val: {strongest_val:.4f})")

        # [8] Feature similarity heatmaps
        print(f"\n[8] Feature similarity heatmaps")
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # IID self-similarity
        sim_iid = torch.mm(w_iid, w_iid.T).cpu().numpy()
        axes[0].imshow(sim_iid, cmap='coolwarm', vmin=-1, vmax=1)
        axes[0].set_title("IID Self-Similarity")

        # Non-IID self-similarity
        sim_niid = torch.mm(w_niid, w_niid.T).cpu().numpy()
        axes[1].imshow(sim_niid, cmap='coolwarm', vmin=-1, vmax=1)
        axes[1].set_title("Non-IID Self-Similarity")

        # Cross-model similarity
        axes[2].imshow(cross_sim.cpu().numpy(), cmap='coolwarm', vmin=-1, vmax=1)
        axes[2].set_title("IID vs Non-IID Cross-Similarity")

        plt.tight_layout()
        plt.show()
    else:
        print("SAE models not trained. Skipping dictionary analysis.")
else:
    print("Analysis skipped: Checkpoints not loaded or activations not extracted.")

In [ ]:
from fedmi.env import show_images
show_images()

## Download Results

In [ ]:
import shutil
from IPython.display import FileLink

if os.path.isdir(CHECKPOINT_DIR):
    output_zip = "sae_experiment_results.zip"
    shutil.make_archive(output_zip.replace('.zip', ''), 'zip', CHECKPOINT_DIR)
    print(f"Results zipped to {output_zip}")
    display(FileLink(output_zip))
else:
    print("⚠ No results found in checkpoint directory.")